In [ ]:
!git clone -q https://github.com/mnhhuyen/AI4SE_toxic_comments.git
%cd AI4SE_toxic_comments
!pip install -q -e ".[dev]" tabulate
!python -m pytest -q 2>&1 | tail -5


/content/AI4SE_toxic_comments/AI4SE_toxic_comments
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for toxic-comments (pyproject.toml) ... done
  /usr/local/lib/python3.13/dist-packages/sklearn/multiclass.py:90: UserWarning: Label not 5 is present in all training examples.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
55 passed, 4 warnings in 14.66s


In [ ]:
import sys
import os
from pathlib import Path

# Dynamically resolve and add the repository root and 'src' directory to sys.path
repo_dir = Path("/content/AI4SE_toxic_comments")
if repo_dir.exists():
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))
    src_dir = repo_dir / "src"
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
else:
    # Fallback to current working directory
    cwd = Path.cwd()
    if str(cwd) not in sys.path:
        sys.path.insert(0, str(cwd))
    if (cwd / "src").exists() and str(cwd / "src") not in sys.path:
        sys.path.insert(0, str(cwd / "src"))

import pandas as pd
from toxic_comments.cleaning import process_cleaning, validate_training_data
from toxic_comments.config import ID_COLUMN, LABEL_COLUMNS, PROCESSED_DATA_DIR, RAW_DATA_DIR
from toxic_comments.splits import save_kfold_datasets

clean_path = PROCESSED_DATA_DIR / "train_clean.csv"
if clean_path.exists():
    data = pd.read_csv(clean_path)
else:
    data = process_cleaning(validate_training_data(pd.read_csv(RAW_DATA_DIR / "train.csv")))
    clean_path.parent.mkdir(parents=True, exist_ok=True)
    data.to_csv(clean_path, index=False)

for column in ["comment_text", "comment_light", "comment_heavy"]:
    data[column] = data[column].fillna("").astype(str)
data = data[data["is_empty_heavy"] == 0].reset_index(drop=True)

keep = [ID_COLUMN, "comment_text", "comment_light", "comment_heavy", *LABEL_COLUMNS]
save_kfold_datasets(data[keep], n_splits=5, strategy="stratified")
print(data.shape)


>>> dataset (159,571 rows)
    cleaning level 1 (light) ...
    cleaning level 2 (heavy) ...
    75 rows empty in HEAVY but fine in LIGHT (non-Latin / emoji) -> KEPT, flagged as is_non_latin
    dropped 95 rows with no usable text at all
    170 rows unusable for bag-of-words -> filter on is_empty_heavy for TF-IDF
(159401, 28)


In [ ]:
from pathlib import Path
from toxic_comments.config import K_FOLD_DATA_DIR, RESULTS_DIR
from toxic_comments.train import train_from_folds
from tqdm.auto import tqdm  # Thư viện hiển thị thanh tiến trình (progress bar)

MODELS = [
    "dummy_most_frequent",
    "tfidf_logistic_regression",
    "word_char_logistic_regression",
    "nbsvm",
]

# Sử dụng tqdm để theo dõi tiến độ tổng thể của danh sách model
for name in tqdm(MODELS, desc="Tiến trình chạy các Models", unit="model"):
    if (RESULTS_DIR / f"{name}_fold_metrics.csv").exists():
        tqdm.write(f"⏭️ Đã tồn tại kết quả, bỏ qua: {name}")
        continue

    tqdm.write(f"\n⏳ Đang huấn luyện: {name} (chạy qua 5 folds)...")

    # Lưu ý: Thời gian lâu nhất sẽ nằm ở hàm này do xử lý văn bản
    train_from_folds(
        folds_dir=K_FOLD_DATA_DIR,
        model_name=name,
        max_features=50_000,
        output_dir=Path("/content/models_tmp"),
    )
    tqdm.write(f"✅ Hoàn thành: {name}")

Tiến trình chạy các Models:   0%|          | 0/4 [00:00<?, ?model/s]


⏳ Đang huấn luyện: dummy_most_frequent (chạy qua 5 folds)...
✅ Hoàn thành: dummy_most_frequent

⏳ Đang huấn luyện: tfidf_logistic_regression (chạy qua 5 folds)...
✅ Hoàn thành: tfidf_logistic_regression

⏳ Đang huấn luyện: word_char_logistic_regression (chạy qua 5 folds)...
✅ Hoàn thành: word_char_logistic_regression

⏳ Đang huấn luyện: nbsvm (chạy qua 5 folds)...
✅ Hoàn thành: nbsvm


In [ ]:
from toxic_comments.evaluation import summarize_folds

folds = pd.concat([pd.read_csv(RESULTS_DIR / f"{n}_fold_metrics.csv") for n in MODELS])
summary = summarize_folds(folds)[["micro_f1", "macro_f1", "micro_roc_auc", "macro_roc_auc"]]
print(summary.to_markdown())

per_label = pd.concat([pd.read_csv(RESULTS_DIR / f"{n}_fold_per_label.csv") for n in MODELS])
print("\n### F1 PER LABEL (mean ± std over 5 folds)")
print(per_label.groupby(["model_name", "label"])["f1"].agg(["mean", "std"]).round(4).to_markdown())


| model_name                    |   ('micro_f1', 'mean') |   ('micro_f1', 'std') |   ('macro_f1', 'mean') |   ('macro_f1', 'std') |   ('micro_roc_auc', 'mean') |   ('micro_roc_auc', 'std') |   ('macro_roc_auc', 'mean') |   ('macro_roc_auc', 'std') |
|:------------------------------|-----------------------:|----------------------:|-----------------------:|----------------------:|----------------------------:|---------------------------:|----------------------------:|---------------------------:|
| dummy_most_frequent           |                 0      |                0      |                 0      |                0      |                      0.5    |                     0      |                      0.5    |                     0      |
| nbsvm                         |                 0.7439 |                0.0042 |                 0.6195 |                0.0068 |                      0.9854 |                     0.0006 |                      0.9793 |                     0.0022 |


| So sánh | Chênh lệch | Độ lệch chuẩn | Kết luận |
| :--- | :---: | :---: | :--- |
| nbsvm vs baseline, micro-F1 | +0.0375 | ~0.0045 | ~8 std — chắc chắn thật |
| word_char vs baseline, micro-F1 | +0.0304 | ~0.0050 | ~6 std — chắc chắn thật |
| word_char vs baseline, macro-AUC | +0.0053 | ~0.0014 | ~4 std — thật |
| nbsvm vs word_char, micro-F1 | +0.0071 | ~0.0048 | ~1,4 std — chưa kết luận được |
| word_char vs nbsvm, macro-F1 | +0.0044 | ~0.0080 | hòa, nằm trong nhiễu |
| word_char vs nbsvm, micro-AUC | +0.0018 | ~0.0004 | ~4 std — thật |

Model chọn: word_char_logistic_regression
Thắng cả hai AUC một cách dứt khoát (micro-AUC std chỉ 0.0002 — cực kỳ ổn định), hòa với nbsvm ở macro-F1, và chỉ thua micro-F1 một khoảng nằm trong vùng chưa kết luận được. Vì metric chính thức của Kaggle là mean column-wise ROC-AUC, đây là model để nộp.

Quy ra mức giảm sai số: macro-AUC từ 0.9776 lên 0.9829 nghĩa là giảm 24% phần sai số còn lại so với model gốc



